# 🔊 Speech Enhancement Demo

This notebook walks through the complete speech enhancement pipeline:
1. Load a noisy audio file
2. Visualize the noisy waveform and spectrogram
3. Run the U-Net denoiser
4. Compare noisy vs. enhanced audio
5. Save the enhanced output

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import torch
import soundfile as sf
import matplotlib.pyplot as plt
from IPython.display import Audio, display

from config import Config
from models.enhancer_unet import SpeechEnhancerUNet
from scripts.utils import load_audio, save_audio, compute_snr, compute_si_snr

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Load or Generate a Noisy Audio Sample

In [ ]:
# Option A: Load an existing noisy file
# noisy = load_audio('path/to/noisy.wav', sr=Config.sample_rate)

# Option B: Create a noisy sample from clean audio
import glob, random

clean_files = glob.glob('../data/clean_speech/*.wav')
if not clean_files:
    print('No clean files found. Creating a synthetic demo signal...')
    t = np.linspace(0, 2, Config.sample_rate * 2)
    clean = (0.5 * np.sin(2 * np.pi * 440 * t) + 0.3 * np.sin(2 * np.pi * 880 * t)).astype(np.float32)
else:
    clean = load_audio(random.choice(clean_files), sr=Config.sample_rate)
    clean = clean[:Config.sample_rate * 3]  # 3 seconds

# Add noise at 5 dB SNR
snr_db = 5.0
sig_power = np.var(clean).clip(1e-8)
noise_power = sig_power / (10 ** (snr_db / 10))
noise = np.random.randn(len(clean)).astype(np.float32)
noise *= np.sqrt(noise_power / np.var(noise).clip(1e-8))
noisy = np.clip(clean + noise, -1, 1).astype(np.float32)

print(f'Clean:  {len(clean)/Config.sample_rate:.2f}s')
print(f'Noisy:  SNR = {snr_db} dB')

## 2. Visualize Noisy Audio

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6))

t = np.arange(len(noisy)) / Config.sample_rate
ax1.plot(t, noisy, color='#ff6b6b', linewidth=0.5)
ax1.set_title('Noisy Waveform', fontsize=12, fontweight='bold')
ax1.set_ylabel('Amplitude')
ax1.set_ylim(-1.1, 1.1)

ax2.specgram(noisy, Fs=Config.sample_rate, NFFT=512, noverlap=384, cmap='magma')
ax2.set_title('Noisy Spectrogram', fontsize=12, fontweight='bold')
ax2.set_ylabel('Frequency (Hz)')
ax2.set_xlabel('Time (s)')

plt.tight_layout()
plt.show()

print('Listen to the noisy audio:')
display(Audio(noisy, rate=Config.sample_rate))

## 3. Load the U-Net Model and Enhance

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = SpeechEnhancerUNet(base_ch=Config.base_channels).to(device)

checkpoint_path = '../checkpoints/best_discriminative.pt'
import os
if os.path.exists(checkpoint_path):
    model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
    print(f'Loaded checkpoint: {checkpoint_path}')
else:
    print('No checkpoint found — using untrained model (output will be noisy)')

model.eval()
num_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {num_params:,}')

In [ ]:
# Process in chunks (handles arbitrary-length audio)
chunk_len = Config.segment_length
enhanced_chunks = []

n_chunks = int(np.ceil(len(noisy) / chunk_len))
for i in range(n_chunks):
    start = i * chunk_len
    end = min(start + chunk_len, len(noisy))
    chunk = noisy[start:end]
    
    pad = chunk_len - len(chunk)
    if pad > 0:
        chunk = np.concatenate([chunk, np.zeros(pad, dtype=np.float32)])
    
    inp = torch.from_numpy(chunk).float().unsqueeze(0).unsqueeze(0).to(device)
    with torch.no_grad():
        out = model(inp)
    out_np = out.cpu().squeeze().numpy()
    
    if pad > 0:
        out_np = out_np[:-pad]
    enhanced_chunks.append(out_np)

enhanced = np.concatenate(enhanced_chunks).astype(np.float32)
enhanced /= np.abs(enhanced).max() + 1e-8

print(f'Enhanced audio: {len(enhanced)/Config.sample_rate:.2f}s')

## 4. Compare: Noisy vs Enhanced

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

t = np.arange(len(noisy)) / Config.sample_rate

axes[0, 0].plot(t, noisy, color='#ff6b6b', linewidth=0.5)
axes[0, 0].set_title('Noisy Waveform', fontweight='bold', color='#ff6b6b')
axes[0, 0].set_ylim(-1.1, 1.1)

axes[0, 1].plot(t, enhanced, color='#00d4aa', linewidth=0.5)
axes[0, 1].set_title('Enhanced Waveform', fontweight='bold', color='#00d4aa')
axes[0, 1].set_ylim(-1.1, 1.1)

axes[1, 0].specgram(noisy, Fs=Config.sample_rate, NFFT=512, noverlap=384, cmap='magma')
axes[1, 0].set_title('Noisy Spectrogram', fontweight='bold', color='#ff6b6b')

axes[1, 1].specgram(enhanced, Fs=Config.sample_rate, NFFT=512, noverlap=384, cmap='magma')
axes[1, 1].set_title('Enhanced Spectrogram', fontweight='bold', color='#00d4aa')

plt.suptitle('Speech Enhancement — Before vs After', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Metrics
snr_noisy = compute_snr(clean, noisy)
snr_enhanced = compute_snr(clean, enhanced)
si_snr_enhanced = compute_si_snr(clean, enhanced)

print(f'SNR noisy:       {snr_noisy:.2f} dB')
print(f'SNR enhanced:    {snr_enhanced:.2f} dB')
print(f'SI-SNR enhanced: {si_snr_enhanced:.2f} dB')
print(f'SNR improvement: +{snr_enhanced - snr_noisy:.2f} dB')

In [ ]:
print('Noisy audio:')
display(Audio(noisy, rate=Config.sample_rate))

print('Enhanced audio:')
display(Audio(enhanced, rate=Config.sample_rate))

## 5. Save Enhanced Audio

In [ ]:
output_path = '../results/audio_samples/enhanced_demo.wav'
os.makedirs(os.path.dirname(output_path), exist_ok=True)
save_audio(output_path, enhanced, sr=Config.sample_rate)
print(f'Saved enhanced audio: {output_path}')

# Also save the noisy version for reference
save_audio('../results/audio_samples/noisy_demo.wav', noisy, sr=Config.sample_rate)
print('Saved noisy reference: ../results/audio_samples/noisy_demo.wav')